In [ ]:
# Preamble
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

## 01 Preparing Data for Classification

In [ ]:
df = pd.read_csv(
    "https://drive.switch.ch/index.php/s/R9OALF0veVCJ4Ru/download",
    usecols=[1, 2],
    names=["product", "description"],
    skiprows=1,
    dtype={"product": "category", "description": "str"},
)

df.head()

In [ ]:
(df.groupby("product")["description"].count().plot(kind="bar", ylim=0))

plt.show()

## 02 Build the text representation

### Split into **Train** and **Test** set

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, test_df = train_test_split(
    df, stratify=df["product"], shuffle=True, random_state=23
)

In [ ]:
# Plot of Product distribution in the train split
(train_df.groupby("product")["description"].count().plot(kind="bar", ylim=0))

plt.show()

In [ ]:
# Plot of Product distribution in the train split
(test_df.groupby("product")["description"].count().plot(kind="bar", ylim=0))

plt.show()

In [ ]:
train_df["description"].sample(1).iloc[0]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer()
X_train = vec.fit_transform(train_df["description"])
print(X_train.shape)

In [ ]:
len(train_df)

In [ ]:
# Get a word
vec.get_feature_names_out()[0b101010101010]

In [ ]:
X_train[2275, 0xAAA]

In [ ]:
train_df.iloc[2275]["description"]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer()
X_train = vec.fit_transform(train_df["description"])

In [ ]:
X_train.shape

In [ ]:
X_train[2275, 0xAAA]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Generate TF-IDF matrix
vectorizer = TfidfVectorizer(max_df=0.90, min_df=5, stop_words="english")

X_train = vectorizer.fit_transform(train_df["description"])
X_test = vectorizer.transform(test_df["description"])

featurenames = list(vectorizer.get_feature_names_out())

print("Sample feature names identified:", random.sample(featurenames, 15))

print("Size of TFIDF matrix for TRAIN:", X_train.shape)
print("Size of TFIDF matrix for TEST:", X_test.shape)

## 03 Building the model

In [ ]:
# Create Labels and integer classes
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(sorted(df["product"].unique()))

print("Classes found:", le.classes_)

# Convert classes to integers for use with ML
y_train = le.transform(train_df["product"])
y_test = le.transform(test_df["product"])
print("Classes converted to integers: ", y_train)


# Build the model
classifier = MultinomialNB().fit(X_train, y_train)

In [ ]:
le.classes_

## 04 Running Predictions

In [ ]:
from sklearn import metrics

print("Testing with Test Data :\n------------------------")
# Predict on test data
predictions = classifier.predict(X_test)

print("Confusion Matrix:")
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(
    y_true=y_test,
    y_pred=predictions,
    normalize="pred",
    ax=ax,
    display_labels=le.classes_,
    xticks_rotation="vertical",
    cmap="RdYlGn",
    text_kw={"fontsize": 10, "color": "black"},
)
plt.show()

print()
print(f"Prediction Accuracy: {metrics.accuracy_score(y_test, predictions):.4f}")

In [ ]:
print(
    classification_report(
        y_test, predictions, target_names=le.classes_, zero_division=0
    )
)

* **micro average** says the function to compute f1 (P,R) by considering total true positives, false negatives and false positives (no matter of the prediction for each label in the dataset)
* **macro average** says the function to compute f1 (P,R) for each label, and returns the average without considering the proportion for each label in the dataset.
* **weighted average** says the function to compute f1 (P,R) for each label, and returns the average considering the proportion for each label in the dataset.

 * **macro avg** = (0.65+0.07+0.62+0.83+0.75+0.00+0.78+0.00+0.00+0.00+0.49)/11 => **0.3809**
 * **weighted avg** = 0.65 x 91/987+0.07 x 56/987+0.62 x 114/987+0.83 x 190/987+0.75 x 242/987+0.00 x 9/987+0.78 x 200/987+0.00 x 2/987+0.00 x 10/987+0.00 x 9/987+0.49 x 64/987 => **0.669**

## A Classifier for brexit.csv


In [ ]:
df = pd.read_csv(
    "https://drive.switch.ch/index.php/s/IaiPF4XKst5uzm5/download",
    dtype={"Label": "category", "Tweet": "str"},
)

df.head()

In [ ]:
(df.groupby("Label")["Tweet"].count().plot(kind="bar", ylim=0))

plt.show()

## Split the data

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, stratify=df["Label"], shuffle=True, random_state=23
)

## Plot

In [ ]:
# Plot of Product distribution in the train split
(train_df.groupby("Label")["Tweet"].count().plot(kind="bar", ylim=0))

plt.show()

In [ ]:
# Plot of Product distribution in the train split
(test_df.groupby("Label")["Tweet"].count().plot(kind="bar", ylim=0))

plt.show()

## Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Generate TF-IDF matrix
vectorizer = TfidfVectorizer(stop_words="english", min_df=3, max_df=0.90)

X_train = vectorizer.fit_transform(train_df["Tweet"])
X_test = vectorizer.transform(test_df["Tweet"])

print(
    "Sample feature names identified:",
    random.sample(sorted(vectorizer.get_feature_names_out()), 15),
)

print("Size of TFIDF matrix for TRAIN:", X_train.shape)
print("Size of TFIDF matrix for TEST:", X_test.shape)

## Building the model

In [ ]:
# Create Labels and integer classes
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(sorted(df["Label"].unique()))

print("Classes found:", le.classes_)

# Convert classes to integers for use with ML
y_train = le.transform(train_df["Label"])
y_test = le.transform(test_df["Label"])
# print("Classes converted to integers: ", y_train)


# Build the model
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

## Running Predictions

In [ ]:
from sklearn import metrics

print("Testing with Test Data :\n------------------------")
# Predict on test data
predictions = classifier.predict(X_test)

print("Confusion Matrix:")
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(
    y_true=y_test,
    y_pred=predictions,
    normalize="pred",
    ax=ax,
    display_labels=le.classes_,
    xticks_rotation="vertical",
    cmap="RdYlGn",
    text_kw={"fontsize": 18, "color": "black"},
)
plt.show()

print()
print(f"Prediction Accuracy: {metrics.accuracy_score(y_test, predictions):.4f}")

In [ ]:
print(classification_report(y_test, predictions, target_names=le.classes_))

In [ ]:
# Try another CLF
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression()
classifier.fit(X_train, y_train)

print(
    classification_report(y_test, classifier.predict(X_test), target_names=le.classes_)
)

In [ ]:
# Try another CLF
from sklearn.svm import LinearSVC

classifier = LinearSVC()
classifier.fit(X_train, y_train)

print(
    classification_report(y_test, classifier.predict(X_test), target_names=le.classes_)
)

In [ ]:
# Try another CLF
from sklearn.neighbors import KNeighborsClassifier

classifier = KNeighborsClassifier()
classifier.fit(X_train, y_train)

print(
    classification_report(y_test, classifier.predict(X_test), target_names=le.classes_)
)

In [ ]:
# Try another CLF
from sklearn.ensemble import AdaBoostClassifier

classifier = AdaBoostClassifier()
classifier.fit(X_train, y_train)

print(
    classification_report(y_test, classifier.predict(X_test), target_names=le.classes_)
)

In [ ]:
# Try another CLF
from sklearn.linear_model import PassiveAggressiveClassifier

classifier = PassiveAggressiveClassifier()
classifier.fit(X_train, y_train)

print(
    classification_report(y_test, classifier.predict(X_test), target_names=le.classes_)
)

In [ ]:
# Try another CLF
from sklearn.linear_model import SGDClassifier

classifier = SGDClassifier()
classifier.fit(X_train, y_train)

print(
    classification_report(y_test, classifier.predict(X_test), target_names=le.classes_)
)

In [ ]:
# Try another CLF
from sklearn.neural_network import MLPClassifier

classifier = MLPClassifier(hidden_layer_sizes=[64, 64], solver="lbfgs")
classifier.fit(X_train, y_train)

print(
    classification_report(y_test, classifier.predict(X_test), target_names=le.classes_)
)

In [ ]:
# Use Dummy to validate your results
from sklearn.dummy import DummyClassifier

classifier = DummyClassifier(
    strategy="uniform"
)  # select each class with same probability
classifier.fit(X_train, y_train)

print(
    classification_report(
        y_test, classifier.predict(X_test), target_names=le.classes_, zero_division=0
    )
)

In [ ]:
# Use Dummy to validate your results
from sklearn.dummy import DummyClassifier

classifier = DummyClassifier(
    strategy="most_frequent"
)  # always select most frequent class from y_train
classifier.fit(X_train, y_train)

print(
    classification_report(
        y_test, classifier.predict(X_test), target_names=le.classes_, zero_division=0
    )
)

In [ ]:
# Use Dummy to validate your results
from sklearn.dummy import DummyClassifier

classifier = DummyClassifier(
    strategy="stratified"
)  # select randomly but with empirical evidence from y_train
classifier.fit(X_train, y_train)

print(
    classification_report(
        y_test, classifier.predict(X_test), target_names=le.classes_, zero_division=0
    )
)